#### platform (test)

In [ ]:
///- --test static --symbol

:< testing

In [ ]:
:< rust_operators
:< unsupported

## fsharp

### os_platform

In [ ]:
:! os_platform' = $'System.Runtime.InteropServices.OSPlatform'

:| os_platform =
    | FreeBSD
    | Linux
    | OSX
    | Windows

:= os_platform = \|
    | FreeBSD => $'`os_platform'.FreeBSD' : os_platform'
    | Linux => $'`os_platform'.Linux' : os_platform'
    | OSX => $'`os_platform'.OSX' : os_platform'
    | Windows => $'`os_platform'.Windows' : os_platform'

:= run_platform \* t. (fn : os_platform -> (() -> t)) : t =
    := result = dyn !+
    $'let mutable _run_platform_!result : `t option = None '
    $'\n##if _FREEBSD'
    fn FreeBSD () |> emit_unit
    $'##endif\n##if _LINUX'
    fn Linux () |> emit_unit
    $'##endif\n##if _OSX'
    fn OSX () |> emit_unit
    $'##endif\n##if _WINDOWS'
    fn Windows () |> emit_unit
    $'##endif'
    $'|> fun x -> _run_platform_!result <- Some x'
    $'match _run_platform_!result with Some x -> x | None -> failwith "platform.run_platform / \
        _run_platform_!result=None"'

:= is_os_platform (x : os_platform') : bool =
    x |> $'System.Runtime.InteropServices.RuntimeInformation.IsOSPlatform'

:= is_windows' () : bool =
    run_platform \|
        | Windows => \ () => !+
        | _ => \ () => !-

## platform

In [ ]:
:= rust_is_windows () : bool = $'cfg\!(windows)' : bool

:= fsharp_is_windows () : bool = Windows |> os_platform |> is_os_platform

:= gleam_os_type_external () =
    global "import gleam/erlang/atom"
    global "@external(erlang, \"os\", \"type\") fn platform_os_type() -> #(atom.Atom, atom.Atom)"

:= gleam_is_windows () : bool =
    backend_switch {
        Gleam = \ () => gleam_os_type_external ()
    }
    $'atom.to_string(platform_os_type().0) == "win32"' : bool

:= zig_is_windows () : bool = $'(@@import("builtin").os.tag == .windows)' : bool

:= no_is_windows_arm () : bool = failwith "platform.is_windows / no arm for this target"

:= is_windows () : bool =
    run_target \|
        | Rust => rust_is_windows
        | Fsharp => fsharp_is_windows
        | Gleam => gleam_is_windows
        | Zig => zig_is_windows
        | _ => no_is_windows_arm

:= get_executable_suffix () =
    ? is_windows ()
    ?> ".exe"
    |: ""

#### get_executable_suffix (test)

In [1]:
///- --test --symbol

get_executable_suffix ()

In [ ]:
:= gleam_os_command_output (command : string) : string =
    := output = erlang_term::apply "os" "cmd" [ erlang_term::charlist command ]
    erlang_term::apply "unicode" "characters_to_binary" [ output ] |> erlang_term::cast

:= os_command_output (command : string) : string =
    run_target \|
        | Gleam => \ () => gleam_os_command_output command
        | _ => \ () => no_arm "os_command_output"

#### an_os_command_returns_its_output (test)

In [2]:
///- --test --symbol
///> gleam

os_command_output "echo spiral" |> sm'::trim |> _assert_eq "spiral"

__assert_eq { expected = spiral }

## main

In [3]:
///> fsharp

:= main () =
    $'let is_windows () = !is_windows ()' : ()
    $'let get_executable_suffix () = !get_executable_suffix ()' : ()